In [1]:
import cv2
import json
from pathlib import Path
import numpy as np

In [2]:
config_path = Path("hud_regions.json")

In [3]:
if config_path.exists():
    with open(config_path, "r") as f:
        hud_config = json.load(f)
else:
    hud_config = {
        "resolution": [3840, 1600],
        "screens": {
            "gameplay": {},
            "spectate": {},
            "lobby": {}
        }
    }

In [4]:
# ==========================================
# SCREEN IDENTIFIER TEMPLATES
# ==========================================

import cv2
import json
from pathlib import Path

source_dir = Path("screens")
template_dir = Path("screensID")
template_dir.mkdir(parents=True, exist_ok=True)

screen_names = [
    "inventory",
    "map",
    "spectate",
    "summary",
    "deathbox"
]

screens = hud_config.setdefault("screens", {})

for screen_name in screen_names:

    image_path = source_dir / f"{screen_name}.png"
    image = cv2.imread(str(image_path))

    if image is None:
        raise FileNotFoundError(image_path)

    # Select one distinctive, fixed-position UI element
    x, y, w, h = map(int, cv2.selectROI(
        f"Select identifier: {screen_name}",
        image,
        showCrosshair=True,
        fromCenter=False
    ))

    cv2.destroyAllWindows()

    if w <= 0 or h <= 0:
        raise ValueError(f"ROI cancelled: {screen_name}")

    crop = image[y:y+h, x:x+w]

    # Save identifier template
    output_path = template_dir / f"{screen_name}.png"

    if not cv2.imwrite(str(output_path), crop):
        raise RuntimeError(f"Could not save {output_path}")

    # Register screen without deleting existing fields
    screen = screens.setdefault(screen_name, {})

    screen["screen_identifier"] = {
        "roi": [x, y, w, h],
        "template": output_path.as_posix(),
        "threshold": 0.90
    }

    # Save after each screen
    with open(config_path, "w") as f:
        json.dump(hud_config, f, indent=2)

    print(f"{screen_name}: ROI={[x, y, w, h]}")

print("All five screen identifiers saved.")

QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.

inventory: ROI=[1535, 28, 273, 70]
map: ROI=[3300, 789, 371, 61]
spectate: ROI=[63, 1520, 801, 67]


QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0
QFont::setPointSizeF: Point size <= 0 (-0.750000), must be greater than 0


summary: ROI=[1786, 10, 267, 100]
deathbox: ROI=[673, 163, 239, 45]
All five screen identifiers saved.


In [4]:
# Legend templates for self, tm1, tm2
source_dir = Path("firingRangeLegends") # change
template_dir = Path("legends") # change
gameplay = hud_config["screens"]["gameplay"]
images = sorted(source_dir.glob("*.png"))
assert images, "No source images found"

reference_image = cv2.imread(str(images[0]))
assert reference_image is not None

for player_name in ["self", "tm1", "tm2"]:

    player = gameplay.setdefault(player_name, {})

    # Select ROI once per player
    x, y, w, h = map(int, cv2.selectROI(
        f"Select {player_name} legend",
        reference_image,
        showCrosshair=True,
        fromCenter=False
    ))
    cv2.destroyAllWindows()

    if w <= 0 or h <= 0:
        raise ValueError(f"ROI selection cancelled: {player_name}")

    # Separate template directory for each player
    output_dir = Path(f"{template_dir}_{player_name}")
    output_dir.mkdir(parents=True, exist_ok=True)

    templates = {}

    # Crop every reference screenshot using this player's ROI
    for path in images:
        image = cv2.imread(str(path))

        if image is None:
            continue

        crop = image[y:y+h, x:x+w]

        if crop.shape[:2] != (h, w):
            raise ValueError(f"Invalid crop: {path.name}")

        output = output_dir / path.name
        cv2.imwrite(str(output), crop)

        templates[path.stem] = str(output)

    player["legend"] = {
        "roi": [x, y, w, h],
        "templates": templates
    }

    # Save after each player
    with open(config_path, "w") as f:
        json.dump(hud_config, f, indent=2)

    print(f"{player_name}: saved {len(templates)} templates")

print("Finished.")

QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.

self: saved 28 templates
tm1: saved 28 templates
tm2: saved 28 templates
Finished.


In [4]:
source_dir = Path("firingRangeHealthSheild")

shield_types = {
    "white": 50,
    "whiteAmped": 75,
    "blue": 75,
    "blueAmped": 100,
    "purple": 100,
    "purpleAmped": 125
}

gameplay = hud_config["screens"]["gameplay"]

for player_name in ["self", "tm1", "tm2"]:

    player = gameplay.setdefault(player_name, {})
    shield = player.setdefault("shield", {})
    types = shield.setdefault("types", {})

    for name, max_value in shield_types.items():

        image = cv2.imread(str(source_dir / f"{name}.png"))

        if image is None:
            print(f"Missing: {name}.png")
            continue

        # Select ROI for this player and shield type
        x, y, w, h = map(int, cv2.selectROI(
            f"{player_name} - {name}",
            image,
            showCrosshair=True,
            fromCenter=False
        ))
        cv2.destroyAllWindows()

        if w <= 0 or h <= 0:
            print(f"Skipped: {player_name}/{name}")
            continue

        crop = image[y:y+h, x:x+w]
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

        median_hsv = np.median(
            hsv.reshape(-1, 3),
            axis=0
        ).round().astype(int).tolist()

        # Update only this player's shield type
        entry = types.setdefault(name, {})
        entry["roi"] = [x, y, w, h]
        entry["max"] = max_value
        entry["median_hsv"] = median_hsv

        # Save after every selection
        with open(config_path, "w") as f:
            json.dump(hud_config, f, indent=2)

        print(
            f"{player_name}/{name}: "
            f"ROI={[x, y, w, h]}, HSV={median_hsv}"
        )

print("All shield configurations saved.")

QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/tony/toji/lib/python3.12/site-packages/cv2/qt/fonts.

self/white: ROI=[263, 1477, 140, 12], HSV=[0, 0, 225]
self/whiteAmped: ROI=[263, 1476, 208, 13], HSV=[0, 0, 235]
self/blue: ROI=[263, 1477, 208, 12], HSV=[91, 172, 255]
self/blueAmped: ROI=[263, 1475, 279, 15], HSV=[90, 166, 255]
self/purple: ROI=[263, 1475, 277, 14], HSV=[145, 71, 255]
self/purpleAmped: ROI=[263, 1475, 347, 14], HSV=[145, 65, 255]
tm1/white: ROI=[206, 1302, 95, 6], HSV=[0, 0, 214]
tm1/whiteAmped: ROI=[206, 1301, 138, 7], HSV=[0, 0, 226]
tm1/blue: ROI=[206, 1302, 141, 6], HSV=[95, 183, 255]
tm1/blueAmped: ROI=[206, 1302, 184, 6], HSV=[94, 176, 255]
tm1/purple: ROI=[206, 1302, 186, 7], HSV=[146, 82, 255]
tm1/purpleAmped: ROI=[206, 1302, 230, 7], HSV=[150, 78, 255]
tm2/white: ROI=[206, 1204, 92, 7], HSV=[0, 0, 210]
tm2/whiteAmped: ROI=[206, 1203, 137, 7], HSV=[0, 0, 232]
tm2/blue: ROI=[206, 1202, 141, 9], HSV=[91, 168, 255]
tm2/blueAmped: ROI=[206, 1203, 180, 7], HSV=[92, 171, 255]
tm2/purple: ROI=[206, 1203, 181, 8], HSV=[150, 72, 255]
tm2/purpleAmped: ROI=[206, 1204, 2

In [11]:
# ==========================================
# WEAPON HUD — 3 ROIs
# ==========================================

source_dir = Path("guns")
template_dir = Path("gunsInfo")
template_dir.mkdir(parents=True, exist_ok=True)

image = cv2.imread(str(source_dir / "weapons.png"))
assert image is not None, "Missing guns/weapons.png"

weapons = (
    hud_config["screens"]["gameplay"]
    .setdefault("self", {})
    .setdefault("weapons", {})
)

regions = {
    "names": "Select BOTH weapon names + slot numbers",
    "magazine": "Select magazine ammo number",
    "reserve": "Select reserve ammo number"
}

for name, instruction in regions.items():

    x, y, w, h = map(int, cv2.selectROI(
        instruction,
        image,
        showCrosshair=True,
        fromCenter=False
    ))
    cv2.destroyAllWindows()

    if w <= 0 or h <= 0:
        raise ValueError(f"ROI cancelled: {name}")

    crop = image[y:y+h, x:x+w]

    output = template_dir / f"{name}.png"
    if not cv2.imwrite(str(output), crop):
        raise RuntimeError(f"Failed to save {output}")

    weapons[name] = {
        "roi": [x, y, w, h],
        "template": output.as_posix()
    }

    with open(config_path, "w") as f:
        json.dump(hud_config, f, indent=2)

    print(f"{name}: ROI={[x, y, w, h]}")

print("All 3 weapon regions saved.")

names: ROI=[3236, 1528, 486, 40]
magazine: ROI=[3537, 1421, 210, 60]
reserve: ROI=[3537, 1467, 207, 56]
All 3 weapon regions saved.
